In [3]:
import os
import json
import pandas as pd
import traceback

In [10]:
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
openai_key=os.getenv("OPENAI_API")

In [11]:
llm2=ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=openai_key
)
llm = ChatOllama(
    model ="llama3.1:latest",
    temperature=1
)

In [14]:
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.callbacks.manager import get_openai_callback
import PyPDF2

C:\Users\maaz7\AppData\Local\Temp\ipykernel_30636\3186980045.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.callbacks.manager import get_openai_callback


In [15]:
from langchain_core.output_parsers import StrOutputParser

In [18]:
Response_json = {
    "1":{
        "mcq":"multiple choice question",
        "options":{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",
            "d":"choice here",
        },
        "correct":"correct answer",
    },
    "2":{
        "mcq":"multiple choice question",
        "options":{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",
            "d":"choice here",
        },
        "correct":"correct answer",
    },
    "3":{
        "mcq":"multiple choice question",
        "options":{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",
            "d":"choice here",
        },
        "correct":"correct answer",
    },
}

In [19]:
template = """
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create quiz of {number} multiple choice for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Ensure to make {number} MCQs
### Response_json
{response_json}

"""

In [20]:
quiz_prompt = PromptTemplate(
    input_variables=["text","number","subject","tone","response_json"],
    template=template
)

In [26]:
quizreview_chain = quiz_prompt | llm | StrOutputParser()

In [27]:
template2= """ 
You are an expert in english grammer and writer. Given a Multiple CHoice Quiz for {subject} students. \
You need to evaluate the complexity of the question and give a complexity analysis of the quiz. Only use at max 50 words for complexity, 
if the quiz is not as per with the cognitive and analytical abilities of the students, \
update the quiz questions which need to be changed and change the tone such that it fits perfectly with the students.
Quiz_mcq:
{quiz}

check from an expert English writer of the above quiz:
"""

In [23]:
quiz_eval_prompt = PromptTemplate(
    input_variables=["subject","quiz"],
    template=template2
)

In [28]:
quizeval_chain = quiz_eval_prompt | llm

In [29]:
from langchain_core.runnables import RunnableLambda

In [30]:
final_chain = quizreview_chain | RunnableLambda( lambda x: {"subject": x,"quiz" : x}) | quizeval_chain | StrOutputParser()

In [32]:
file_path = r"C:\Users\maaz7\mcqgenerator\data.txt"

In [33]:
with open(file_path, 'r') as file:
    text = file.read()

In [34]:
print(text)

A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contexts, and are a foundational technology behind modern chatbots.[1] Biased or inaccurate training data can make an LLM's output less reliable.[2]

LLMs are typically based on transformer architecture, which is more parallelizable than earlier recurrent neural network models.[3] Generative pre-trained transformers (GPTs) are a type of LLM that is pre-trained to predict the next word.[4] GPTs are then often fine-tuned to follow instructions and to behave as assistants.[5]

Benchmark evaluations for LLMs attempt to measure model reasoning, factual accuracy, alignment, and safety.[6]

History

The number of publications about large language models by year grouped by publication types

The training compute of notable large models in FLOPs vs publication date over

In [35]:
json.dumps(Response_json)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [ ]:
#how to check token usage 
with get_openai_callback() as cb:
    response=final_chain(
        {
            "text":text,
            "number":number,
            "subject":subject,
            "tone":tone,
            "response_json":json.dumps(Response_json)
        }
    )

NameError: name 'number' is not defined